In [1]:
import sys
import numpy as np
import os
sys.path.append(os.getcwd())
import scipy as sp
import scipy.sparse as spsp
from scipy.sparse import csr_matrix, coo_matrix
import itertools
from fast_inversion.BPX import FEM_BPX_helpers as FEM

In [7]:
D = 2 # spatial dimensions
L = 3 # discretization level

$C_{M,v} = \left( \bigotimes_{i=0}^{d} R_{L,1D} \right)$
- $\quad C_{M,v}$ is a $2^{d(L+1)} \times (2^L + 1)^d$ matrix

and

$R_{L,1D} = 2^{-L/2} \begin{bmatrix}
        \frac{1}{2} & \frac{1}{2} & 0 & \cdots & 0  \\
        -\frac{1}{2 \sqrt{3}} & \frac{1}{2 \sqrt{3}} & 0 & \cdots & 0 \\
        0 & \frac{1}{2} & \frac{1}{2} & \cdots & 0 \\
        0 & -\frac{1}{2 \sqrt{3}} & \frac{1}{2 \sqrt{3}} & \cdots & 0 \\
        0 & 0 & \frac{1}{2} & \cdots & 0 \\
        0 & 0 & -\frac{1}{2 \sqrt{3}} & \cdots & 0 \\
        \vdots & \vdots & \vdots & \ddots & 0 \\
        0 & 0 & 0 & \cdots & \frac{1}{2} \\
        0 & 0 & 0 & \cdots & \frac{1}{2 \sqrt{3}} \\
    \end{bmatrix} = 2^{-L/2}(I_{2^L} \otimes \begin{bmatrix}
        \frac{1}{2} &  \frac{1}{2} \\
        -\frac{1}{2 \sqrt{3}} &  \frac{1}{2 \sqrt{3}} \\
    \end{bmatrix}) M_2 $
- $\quad R_{L,1D}$ is a $2^{L+1} \times 2^L + 1$ matrix

$M_2 = \begin{bmatrix}
        1 & 0 & 0 & \cdots & 0 & 0 \\
        0 & 1 & 0 & \cdots & 0 & 0 \\
        0 & 1 & 0 & \cdots & 0 & 0 \\
        0 & 0 & 1 & \cdots & 0 & 0 \\
        0 & 0 & 1 & \cdots & 0 & 0 \\
        \vdots & \vdots & \vdots & \ddots & 0 & 0 \\
        0 & 0 & 0 & \cdots & 1 & 0 \\
        0 & 0 & 0 & \cdots & 1 & 0 \\
        0 & 0 & 0 & \cdots & 0 & 1 \\
    \end{bmatrix}$

In [15]:
C_m_v = np.array(FEM.getC_m_v(D, L)) # Vacuum C_m matrix

abs_matrix = np.arange(1, 2**(L*D)+1).reshape([2**L] * D) # absorption matrix with incrementally increasing absorption coefficients
#abs_matrix = np.ones(2**(L*D)).reshape([2**L] * D) # absorption matrix with all ones absorption coefficients

Sigma_a = np.kron(np.diag(abs_matrix.flatten()), np.eye(int(2**D))) # diagonal matrix of absorption coefficients

A_v_test = C_m_v.T @ Sigma_a @ C_m_v # Use the Deiml matrices to make the Vacuum mass matrix

A_v = FEM.get_mass_matrix_v_brute_force(L, D, abs_matrix).toarray() # Use the brute force functions to make the Vacuum mass matrix
A_v_error = A_v - A_v_test
print("L2 norm of A_v_error = ", np.linalg.norm(A_v_error, 'fro'))

L2 norm of A_v_error =  9.687679614063946e-16


## Verification of the absorption decomposition in 1D, 2D, and 3D

For a piecewise-constant absorption coefficient, each spatial cell contributes one coefficient to each of its $2^D$ local tensor-product quadrature modes. Therefore

$$\Sigma_a = \operatorname{diag}(\operatorname{vec}(\texttt{abs\_matrix})) \otimes I_{2^D},$$

and the vacuum-boundary absorption matrix is

$$A_{a,v} = C_{m,v}^{T}\Sigma_a C_{m,v}.$$

The following cell compares this decomposition directly with `FEM.get_mass_matrix_v_brute_force` for 1, 2, and 3 spatial dimensions. A spatially varying, positive absorption field is used in every case; the multidimensional fields include both direction-dependent and cross-coordinate variation. The assertions make the cell fail if the two constructions do not agree to floating-point precision.

In [ ]:
L_test = 2
verification_results = []

for D_test in (1, 2, 3):
    cell_shape = (2**L_test,) * D_test
    cell_indices = np.indices(cell_shape)

    # Positive, nonuniform absorption coefficients.  For D > 1, the product
    # term makes the distribution nonseparable as well as spatially varying.
    abs_matrix_test = 0.4 + sum(
        0.15 * (axis + 1)**2 / (direction + 1)
        for direction, axis in enumerate(cell_indices)
    )
    if D_test > 1:
        abs_matrix_test += 0.1 * np.prod(cell_indices + 1, axis=0)

    C_m_v_test = np.asarray(FEM.getC_m_v(D_test, L_test))
    Sigma_a_test = np.kron(
        np.diag(abs_matrix_test.ravel(order="C")),
        np.eye(2**D_test),
    )

    A_from_decomposition = C_m_v_test.T @ Sigma_a_test @ C_m_v_test
    A_from_brute_force = FEM.get_mass_matrix_v_brute_force(
        L_test, D_test, abs_matrix_test
    ).toarray()

    difference = A_from_brute_force - A_from_decomposition
    max_abs_error = np.max(np.abs(difference))
    frobenius_error = np.linalg.norm(difference, ord="fro")
    matrices_match = np.allclose(
        A_from_brute_force,
        A_from_decomposition,
        rtol=1e-12,
        atol=1e-12,
    )

    verification_results.append(
        (D_test, abs_matrix_test.shape, max_abs_error, frobenius_error, matrices_match)
    )
    assert matrices_match, f"Absorption-matrix constructions differ for D={D_test}"

print(" D | abs_matrix shape | max absolute error | Frobenius error | match")
print("---|------------------|--------------------|-----------------|------")
for D_test, shape, max_error, fro_error, match in verification_results:
    print(f" {D_test} | {str(shape):16s} | {max_error:18.3e} | {fro_error:15.3e} | {match}")

 D | abs_matrix shape | max absolute error | Frobenius error | match
---|------------------|--------------------|-----------------|------
 1 | (4,)             |          5.551e-17 |       7.093e-17 | True
 2 | (4, 4)           |          5.551e-17 |       9.014e-17 | True
 3 | (4, 4, 4)        |          2.429e-17 |       5.067e-17 | True
